# 08 — PaddleOCR-VL-1.6 Full SFT with ERNIEKit — Google Colab Pro

This notebook adapts the **official ERNIEKit PaddleOCR-VL SFT recipe** to PaddleOCR-VL-1.6 and UIT-HWDB-line.

Official SFT guide: https://github.com/PaddlePaddle/ERNIE/blob/release/v1.5/docs/paddleocr_vl_sft.md

Important qualification: the published SFT guide/config is for the PaddleOCR-VL 0.9B/v1-era family. PaddleOCR-VL-1.6 is officially stated to be architecture-compatible with v1.5, so this notebook uses the same ERNIEKit interface with the 1.6 HF checkpoint. This is a reasonable compatibility adaptation, but less directly documented than GLM-OCR fine-tuning. A smoke job is mandatory before the full run.

Training policy: **2 epochs first**. Because this is full SFT on only 6,346 training lines, we do not automatically run epoch 3. Notebook 09 compares epoch-1 vs epoch-2 by validation CER; a third epoch should only be considered if validation CER is still improving.

## 0. Install PaddlePaddle + ERNIEKit using the official sequence

In [ ]:
%pip install -q "paddlepaddle-gpu==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%cd /content
!rm -rf ERNIE
!git clone --depth 1 --branch release/v1.5 https://github.com/PaddlePaddle/ERNIE.git
%cd /content/ERNIE
%pip install -q -r requirements/gpu/requirements.txt
%pip install -q -e .
%pip install -q tensorboard opencv-python-headless "numpy==1.26.4" "kagglehub>=1.0.2" "huggingface_hub>=0.34" nvidia-ml-py

> The official guide recommends CUDA 12+ and an official Paddle Docker image. Colab cannot use that Docker workflow, so this notebook is a best-effort manual install. If imports fail after dependency replacement, restart the Colab session once and continue from the next cell.

## 1. Data

In [ ]:
import os, json, time, random, platform, unicodedata, gc, math, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from jiwer import cer, wer

from google.colab import drive
drive.mount('/content/drive')

# Optional Colab secret. KaggleHub can also prompt/authenticate through its normal flow.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

import kagglehub

SEED = 42
RAW_HANDLE = 'ntklinhfitus/uit-hwdb'
MANIFEST_HANDLE = 'ntklinhfitus/uit-hwdb-manifest'
PROJECT_ROOT = Path('/content/drive/MyDrive/vlm_handwriting_ocr')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Try a partial raw download first. Fall back to the full Kaggle dataset if the
# installed KaggleHub/runtime does not accept directory-level download.
try:
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE, path='UIT_HWDB_line'))
except Exception as e:
    print('Partial raw download unavailable, falling back to full dataset:', repr(e))
    raw_download = Path(kagglehub.dataset_download(RAW_HANDLE))
manifest_download = Path(kagglehub.dataset_download(MANIFEST_HANDLE))

def locate_raw_line_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    candidates=[p for p in pool if (p/'train_data').is_dir() and (p/'test_data').is_dir()]
    assert candidates, f'Cannot locate UIT-HWDB-line train_data/test_data under {base}'
    candidates.sort(key=lambda p: ('UIT_HWDB_line' not in str(p), len(str(p))))
    return candidates[0]

def locate_manifest_root(base: Path) -> Path:
    pool=[base]+[p for p in base.rglob('*') if p.is_dir()]
    for p in pool:
        if all((p/f).exists() for f in ['train.csv','val.csv','test.csv']):
            return p
    raise FileNotFoundError(f'Cannot locate train.csv/val.csv/test.csv under {base}')

RAW_ROOT=locate_raw_line_root(raw_download)
MANIFEST_ROOT=locate_manifest_root(manifest_download)
print('RAW_ROOT      =',RAW_ROOT)
print('MANIFEST_ROOT =',MANIFEST_ROOT)

In [ ]:
train_df=pd.read_csv(MANIFEST_ROOT/'train.csv')
val_df=pd.read_csv(MANIFEST_ROOT/'val.csv')
test_df=pd.read_csv(MANIFEST_ROOT/'test.csv')
EXPECTED={'train':6346,'validation':682,'test':201}
assert len(train_df)==EXPECTED['train'],len(train_df)
assert len(val_df)==EXPECTED['validation'],len(val_df)
assert len(test_df)==EXPECTED['test'],len(test_df)
required={'writer_id','filename','relative_path','text'}
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=required-set(frame.columns)
    assert not missing,f'{name} missing columns: {missing}'
train_writers=set(train_df.writer_id); val_writers=set(val_df.writer_id); test_writers=set(test_df.writer_id)
assert train_writers.isdisjoint(val_writers)
assert train_writers.isdisjoint(test_writers)
assert val_writers.isdisjoint(test_writers)

def resolve_image_path(row):
    return RAW_ROOT/str(row['relative_path'])
for name,frame in [('train',train_df),('validation',val_df),('test',test_df)]:
    missing=[str(resolve_image_path(r)) for _,r in frame.iterrows() if not resolve_image_path(r).exists()]
    assert not missing,f'{name}: missing image paths, e.g. {missing[:3]}'
print(f'Train      : {len(train_df)} samples | {len(train_writers)} writers')
print(f'Validation : {len(val_df)} samples | {len(val_writers)} writers')
print(f'Test       : {len(test_df)} samples | {len(test_writers)} writers')
print('✅ Frozen writer-disjoint split verified.')

## 2. GPU profile and two-epoch step plan

In [ ]:
import subprocess,math,torch
from pynvml import nvmlInit,nvmlDeviceGetHandleByIndex,nvmlDeviceGetMemoryInfo
assert torch.cuda.is_available(),'CUDA GPU required for Paddle full SFT.'
BF16=torch.cuda.is_bf16_supported()
if not BF16:
    raise RuntimeError('The official ERNIEKit PaddleOCR-VL SFT config uses BF16. Switch Colab Pro to a BF16-capable GPU (for example L4/A100) or request a team GPU rather than silently changing training precision.')
nvmlInit(); h=nvmlDeviceGetHandleByIndex(0); total_gb=nvmlDeviceGetMemoryInfo(h).total/1024**3
if total_gb>=70: PACKING,GRAD_ACC,MAX_SEQ_LEN=8,8,16384
elif total_gb>=40: PACKING,GRAD_ACC,MAX_SEQ_LEN=4,16,8192
elif total_gb>=24: PACKING,GRAD_ACC,MAX_SEQ_LEN=2,32,8192
else: PACKING,GRAD_ACC,MAX_SEQ_LEN=1,64,4096
EFFECTIVE=PACKING*GRAD_ACC
STEPS_PER_EPOCH=math.ceil(len(train_df)/EFFECTIVE); MAX_STEPS=2*STEPS_PER_EPOCH; SAVE_STEPS=STEPS_PER_EPOCH; WARMUP_STEPS=max(1,round(MAX_STEPS*.01))
print(f'VRAM={total_gb:.1f}GB | packing={PACKING} | grad_acc={GRAD_ACC} | effective samples/update≈{EFFECTIVE} | max_seq_len={MAX_SEQ_LEN}')
print('steps/epoch=',STEPS_PER_EPOCH,'max_steps=',MAX_STEPS,'save_steps=',SAVE_STEPS,'warmup=',WARMUP_STEPS)
if total_gb<24: print('⚠️ Full SFT may still OOM on this GPU. Smoke run will decide whether to request a larger team GPU.')

## 3. Download PaddleOCR-VL-1.6 locally

In [ ]:
from huggingface_hub import snapshot_download
MODEL_ID='PaddlePaddle/PaddleOCR-VL-1.6'
MODEL_DIR=Path('/content/PaddleOCR-VL-1.6')
snapshot_download(MODEL_ID,local_dir=str(MODEL_DIR))
print('MODEL_DIR=',MODEL_DIR)
for required in ['config.json','chat_template.jinja','inference.yml']:
    print(required,(MODEL_DIR/required).exists())

## 4. Convert frozen train/validation CSVs to the official ERNIEKit JSONL format

In [ ]:
DATA_DIR=Path('/content/paddle_sft_data'); DATA_DIR.mkdir(exist_ok=True)
TRAIN_JSONL=DATA_DIR/'uit_hwdb_line_train.jsonl'; VAL_JSONL=DATA_DIR/'uit_hwdb_line_val.jsonl'
def write_jsonl(frame,path):
    with path.open('w',encoding='utf-8') as f:
        for _,row in frame.iterrows():
            obj={'image_info':[{'matched_text_index':0,'image_url':str(resolve_image_path(row))}], 'text_info':[{'text':'OCR:','tag':'mask'},{'text':str(row['text']),'tag':'no_mask'}]}
            f.write(json.dumps(obj,ensure_ascii=False)+'\n')
write_jsonl(train_df,TRAIN_JSONL); write_jsonl(val_df,VAL_JSONL)
print(TRAIN_JSONL,'lines=',sum(1 for _ in TRAIN_JSONL.open())); print(VAL_JSONL,'lines=',sum(1 for _ in VAL_JSONL.open()))
print(TRAIN_JSONL.open(encoding='utf-8').readline()[:500])

## 5. Build command from the official 16k config

In [ ]:
ERNIE_ROOT=Path('/content/ERNIE'); BASE_CONFIG=ERNIE_ROOT/'examples/configs/PaddleOCR-VL/sft/run_ocr_vl_sft_16k.yaml'; assert BASE_CONFIG.exists()
LOCAL_OUTPUT=Path('/content/paddleocr_vl_1_6_sft_full')
FULL_CMD=['erniekit','train',str(BASE_CONFIG),f'model_name_or_path={MODEL_DIR}',f'train_dataset_path={TRAIN_JSONL}',f'eval_dataset_path={VAL_JSONL}',f'output_dir={LOCAL_OUTPUT}',f'logging_dir={LOCAL_OUTPUT}/tensorboard_logs',f'packing_size={PACKING}',f'max_seq_len={MAX_SEQ_LEN}',f'gradient_accumulation_steps={GRAD_ACC}','batch_size=1','learning_rate=5.0e-6','num_train_epochs=2',f'max_steps={MAX_STEPS}',f'save_steps={SAVE_STEPS}',f'warmup_steps={WARMUP_STEPS}','save_total_limit=3','seed=42','compute_type=bf16','recompute=True','convert_from_hf=True','save_to_hf=True']
print(' '.join(map(str,FULL_CMD)))

## 6. Mandatory small full-SFT smoke job

In [ ]:
RUN_SMOKE_TRAINING=True
smoke_train=train_df.sample(n=128,random_state=SEED).reset_index(drop=True); smoke_val=val_df.sample(n=32,random_state=SEED).reset_index(drop=True)
SMOKE_TRAIN=DATA_DIR/'smoke_train.jsonl'; SMOKE_VAL=DATA_DIR/'smoke_val.jsonl'; write_jsonl(smoke_train,SMOKE_TRAIN); write_jsonl(smoke_val,SMOKE_VAL)
SMOKE_OUTPUT=Path('/content/paddleocr_vl_1_6_sft_smoke'); SMOKE_STEPS=max(2,math.ceil(len(smoke_train)/EFFECTIVE))
SMOKE_CMD=['erniekit','train',str(BASE_CONFIG),f'model_name_or_path={MODEL_DIR}',f'train_dataset_path={SMOKE_TRAIN}',f'eval_dataset_path={SMOKE_VAL}',f'output_dir={SMOKE_OUTPUT}',f'logging_dir={SMOKE_OUTPUT}/tensorboard_logs',f'packing_size={PACKING}',f'max_seq_len={MAX_SEQ_LEN}',f'gradient_accumulation_steps={GRAD_ACC}','batch_size=1','learning_rate=5.0e-6','num_train_epochs=1',f'max_steps={SMOKE_STEPS}',f'save_steps={SMOKE_STEPS}',f'warmup_steps={max(1,round(SMOKE_STEPS*.01))}','save_total_limit=1','seed=42','compute_type=bf16','recompute=True','convert_from_hf=True','save_to_hf=True']
print('Smoke steps=',SMOKE_STEPS)

In [ ]:
if RUN_SMOKE_TRAINING:
    os.chdir('/content/ERNIE'); env={**os.environ,'CUDA_VISIBLE_DEVICES':'0'}; rc=subprocess.run(SMOKE_CMD,env=env).returncode; assert rc==0,f'Paddle smoke training failed with code {rc}'
    print('✅ Paddle full-SFT smoke passed:',SMOKE_OUTPUT)
else: print('Smoke skipped.')

## 7. Full 2-epoch SFT — gated
Set `RUN_FULL_TRAINING=True` only after the smoke job completes without OOM. If smoke OOMs, record the measured GPU and request a larger team GPU rather than repeatedly forcing the same job.

In [ ]:
RUN_FULL_TRAINING=False
if RUN_FULL_TRAINING:
    if LOCAL_OUTPUT.exists(): shutil.rmtree(LOCAL_OUTPUT)
    os.chdir('/content/ERNIE'); rc=subprocess.run(FULL_CMD,env={**os.environ,'CUDA_VISIBLE_DEVICES':'0'}).returncode; assert rc==0,f'Full Paddle SFT failed: {rc}'
    print('✅ Full 2-epoch training completed:',LOCAL_OUTPUT)
else: print('Full training gated.')

## 8. Export model-only epoch snapshots to Google Drive

In [ ]:
DRIVE_CKPT_ROOT=PROJECT_ROOT/'checkpoints'/'paddleocr_vl_1_6'; DRIVE_CKPT_ROOT.mkdir(parents=True,exist_ok=True)

def checkpoint_step(p):
    m=re.search(r'checkpoint[-_](\d+)',p.name); return int(m.group(1)) if m else 10**18

def copy_inference_snapshot(src,dst):
    dst.mkdir(parents=True,exist_ok=True)
    exact={'config.json','preprocessor_config.json','generation.json','generation_config.json','tokenizer.model','tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json','processor_config.json','static_name_to_dyg_name.json'}
    prefixes=('model','configuration_','modeling_','image_processing_','processing_')
    copied=[]
    for p in src.iterdir():
        if p.is_file() and (p.name in exact or p.name.startswith(prefixes) or p.suffix=='.safetensors'):
            shutil.copy2(p,dst/p.name); copied.append(p.name)
    for name in ['chat_template.jinja','inference.yml']:
        if (MODEL_DIR/name).exists(): shutil.copy2(MODEL_DIR/name,dst/name); copied.append(name)
    if not any(n.endswith('.safetensors') for n in copied):
        raise RuntimeError(f'No model safetensors copied from {src}. Inspect ERNIEKit output structure before deleting local output.')
    return copied

if RUN_FULL_TRAINING:
    ckpts=sorted([p for p in LOCAL_OUTPUT.rglob('checkpoint-*') if p.is_dir()],key=checkpoint_step)
    print('Detected checkpoint dirs:',ckpts)
    assert len(ckpts)>=2,'Expected at least epoch-1 and epoch-2 checkpoints because save_steps=steps_per_epoch.'
    selected=ckpts[:2]
    for epoch,src in enumerate(selected,1):
        dst=DRIVE_CKPT_ROOT/f'epoch{epoch}';
        if dst.exists(): shutil.rmtree(dst)
        copied=copy_inference_snapshot(src,dst); print(f'epoch{epoch}:',len(copied),'files ->',dst)
else: print('Export waits for full training.')

## 9. Save training provenance

In [ ]:
prov={'model_id':MODEL_ID,'method':'Full SFT via ERNIEKit release/v1.5 recipe adapted to v1.6','epochs_planned':2,'learning_rate':5e-6,'packing_size':PACKING,'gradient_accumulation_steps':GRAD_ACC,'effective_samples_per_update_approx':EFFECTIVE,'max_seq_len':MAX_SEQ_LEN,'steps_per_epoch':STEPS_PER_EPOCH,'max_steps':MAX_STEPS,'save_steps':SAVE_STEPS,'gpu_vram_gb':total_gb,'bf16':BF16,'train_samples':len(train_df),'val_samples':len(val_df),'seed':SEED,'epoch3_policy':'Do not run automatically; consider only if epoch2 validation CER improves over epoch1.'}
(DRIVE_CKPT_ROOT/'training_provenance.json').write_text(json.dumps(prov,ensure_ascii=False,indent=2),encoding='utf-8'); print(json.dumps(prov,indent=2))